## Cell 1: Import thư viện cần thiết

**Mục tiêu của cell này**
- Nạp đầy đủ thư viện để chạy **FP-Growth / Association Rules** theo đúng phạm vi của file 07.
- Bảo đảm notebook có thể:
  1. đọc `transactions_df` từ bước preprocessing,
  2. tạo **frequent itemsets**,
  3. sinh **association rules**,
  4. xuất artifact đủ sạch cho **UI / integration / báo cáo**.

**Liên hệ với rubric và báo cáo**
- Rubric yêu cầu phần FP-Growth phải có: `frequent itemsets`, `association rules`, phân tích `support / confidence / lift`, và có top rules đủ ý nghĩa để trình bày.  
- Báo cáo mẫu yêu cầu phải có code/logic cho `TransactionEncoder`, `fpgrowth`, `association_rules`, số lượng itemsets/rules, và bảng rule tiêu biểu.

**Đầu ra mong đợi sau toàn notebook**
- `frequent_itemsets.csv`
- `association_rules.csv`
- `top_association_rules_preview.csv`
- `top_association_rules_report.csv`
- `fpgrowth_final_summary.json`


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

## Cell 2: Khai báo cấu hình và đọc dữ liệu giao dịch

**Mục tiêu của cell này**
- Xác định đúng đường dẫn dự án, thư mục output, và file nguồn đầu vào cho mining.
- Khai báo toàn bộ threshold/cấu hình theo một chỗ để notebook dễ kiểm soát và không làm lệch contract với notebook khác.

**Ràng buộc thiết kế phải giữ nguyên**
- Notebook này **giữ đúng grain hiện tại theo pipeline đã có**:  
  `order_id -> danh sách product_category_name` ở mức **order-level category basket**.
- **Không đổi grain sang product-level ở file 07** để tránh làm lệch logic đã được lưu từ preprocessing và tránh ảnh hưởng notebook khác.

**Ý nghĩa các nhóm cấu hình**
- `MIN_ITEMS_FOR_MINING`: chỉ giữ basket có ít nhất 2 item để market basket mining có ý nghĩa.
- `ABSOLUTE_SUPPORT_CANDIDATES` và `CONFIDENCE_GRID`: tạo lưới tìm threshold hợp lý thay vì hard-code một giá trị duy nhất.
- `UI_MIN_*`: ngưỡng tối thiểu để rules xuất ra cho UI không quá bẩn.
- `REPORT_MIN_*`: ngưỡng chặt hơn để rules dùng trong báo cáo có ý nghĩa thống kê tốt hơn.

**Đầu ra mong đợi**
- Đọc thành công `transactions_df.parquet` hoặc `transactions_df.csv`
- In ra thông tin nguồn dữ liệu và mẫu 5 dòng đầu để kiểm tra contract đầu vào


In [2]:
# =========================
# CONFIG
# =========================
def locate_project_base():
    candidates = [Path(".."), Path("."), Path("../..")]
    for base in candidates:
        if (base / "data" / "processed").exists():
            return base
    return Path("..")

BASE_DIR = locate_project_base()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
ARTIFACT_DIR = BASE_DIR / "artifacts"

METRIC_DIR = ARTIFACT_DIR / "metrics"
PRED_DIR = ARTIFACT_DIR / "predictions"

for folder in [METRIC_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Mining rule trên order-level category basket:
# - loại basket 1 item
# - loại 'unknown'
# - giữ grain hiện tại theo đúng tài liệu Tuần 2 / báo cáo mẫu
MIN_ITEMS_FOR_MINING = 2

# Grid support được sinh theo số lần xuất hiện tuyệt đối trên tập mining.
# Ví dụ count=2 nghĩa là itemset phải xuất hiện ít nhất 2 transaction.
ABSOLUTE_SUPPORT_CANDIDATES = [2, 3, 5, 8, 10, 15]
CONFIDENCE_GRID = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

# Contract cho UI / report
UI_MIN_LIFT = 1.0
UI_MIN_CONFIDENCE = 0.05
UI_MIN_SUPPORT_COUNT = 2

REPORT_MIN_CONFIDENCE = 0.10
REPORT_MIN_SUPPORT_COUNT = 5

TOP_RULES_BY_LIFT = 10
TOP_RULES_FOR_REPORT = 5
MAX_RULES_TO_SAVE = 500
PREFERRED_MIN_RULES = 5

parquet_path = PROCESSED_DIR / "transactions_df.parquet"
csv_path = PROCESSED_DIR / "transactions_df.csv"

if parquet_path.exists():
    transactions_raw = pd.read_parquet(parquet_path)
    transactions_source_path = parquet_path
elif csv_path.exists():
    transactions_raw = pd.read_csv(csv_path)
    transactions_source_path = csv_path
else:
    raise FileNotFoundError("Không tìm thấy transactions_df.parquet hoặc transactions_df.csv")

print("BASE_DIR:", BASE_DIR.resolve())
print("Đã đọc:", transactions_source_path)
display(Markdown("### 5 dòng đầu của transactions_df (FINAL: order-level category basket)"))
display(transactions_raw.head())
print("Shape:", transactions_raw.shape)
print("Columns:", list(transactions_raw.columns))

BASE_DIR: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project
Đã đọc: ..\data\processed\transactions_df.parquet


### 5 dòng đầu của transactions_df (FINAL: order-level category basket)

,order_id,transaction_items
0,00010242fe8c5a6d1ba2dd792cb16214,[cool_stuff]
1,00018f77f2f0320c557190d7a144bdd3,[pet_shop]
2,000229ec398224ef6ca0657da4fc703e,[furniture_decor]
3,00024acbcdf0a6daa1e931b038114c75,[perfumery]
4,00042b26cf59d7ce69dfabb4e55b4fd9,[garden_tools]


Shape: (96478, 2)
Columns: ['order_id', 'transaction_items']


## Cell 3: Làm sạch và chuẩn hóa danh sách `transaction_items`

**Mục tiêu của cell này**
- Chuẩn hóa cột `transaction_items` về đúng dạng **list item sạch** cho mỗi `order_id`.
- Xử lý an toàn các trường hợp dữ liệu đến từ:
  - parquet (list thật),
  - CSV (string kiểu `['a', 'b']`),
  - chuỗi phân tách bằng dấu phẩy,
  - giá trị rỗng / null / `unknown format`.

**Logic làm sạch**
1. Kiểm tra schema bắt buộc: phải có `order_id` và `transaction_items`.
2. Chuẩn hóa `order_id` về string không rỗng.
3. Parse `transaction_items` về list.
4. Loại item rỗng / null / `nan` / `none`.
5. Khử trùng lặp item trong cùng một order.
6. Nếu có duplicate `order_id`, gộp lại an toàn để không làm hỏng basket.

**Đầu ra mong đợi**
- `transactions_clean_df` với 3 cột:
  - `order_id`
  - `transaction_items`
  - `item_count`
- Không còn transaction rỗng


In [3]:
# =========================
# CLEAN & PARSE TRANSACTIONS
# =========================
required_cols = ["order_id", "transaction_items"]
missing_cols = [c for c in required_cols if c not in transactions_raw.columns]
if missing_cols:
    raise KeyError(f"Thiếu cột bắt buộc trong transactions_df: {missing_cols}")

def normalize_transaction_items(value):
    # parquet: list thật
    if isinstance(value, (list, tuple, set, np.ndarray)):
        items = list(value)

    # csv: string kiểu "['a', 'b']"
    elif isinstance(value, str):
        text = value.strip()

        if text == "" or text.lower() in {"nan", "none", "null"}:
            items = []
        else:
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    items = list(parsed)
                else:
                    items = [str(parsed)]
            except Exception:
                if "," in text:
                    items = [x.strip() for x in text.split(",")]
                else:
                    items = [text]

    elif pd.isna(value):
        items = []
    else:
        items = [str(value)]

    cleaned = []
    for item in items:
        if isinstance(item, (list, tuple, set, np.ndarray)):
            for sub_item in list(item):
                if sub_item is None:
                    continue
                sub_item_str = str(sub_item).strip()
                if sub_item_str == "" or sub_item_str.lower() in {"nan", "none", "null"}:
                    continue
                cleaned.append(sub_item_str)
            continue

        if item is None:
            continue

        item_str = str(item).strip()
        if item_str == "" or item_str.lower() in {"nan", "none", "null"}:
            continue

        cleaned.append(item_str)

    # market basket = set item trong cùng 1 order
    cleaned = sorted(set(cleaned))
    return cleaned

transactions_df = transactions_raw.copy()
transactions_df = transactions_df.dropna(subset=["order_id"]).copy()
transactions_df["order_id"] = transactions_df["order_id"].astype(str).str.strip()
transactions_df = transactions_df[transactions_df["order_id"] != ""].copy()

transactions_df["transaction_items"] = (
    transactions_df["transaction_items"].apply(normalize_transaction_items)
)

transactions_df["item_count"] = transactions_df["transaction_items"].apply(len)

rows_before_clean = len(transactions_df)
dup_orders_before = int(transactions_df["order_id"].duplicated().sum())

# nếu lỡ có duplicated order_id do save lỗi, gộp lại an toàn
if dup_orders_before > 0:
    transactions_df = (
        transactions_df
        .groupby("order_id", as_index=False)
        .agg(
            transaction_items=(
                "transaction_items",
                lambda rows: sorted(set(item for row in rows for item in row))
            )
        )
    )
    transactions_df["item_count"] = transactions_df["transaction_items"].apply(len)

transactions_clean_df = transactions_df[["order_id", "transaction_items", "item_count"]].copy()
transactions_clean_df = transactions_clean_df[
    transactions_clean_df["item_count"] > 0
].copy()

rows_after_clean = len(transactions_clean_df)

display(Markdown("### Cleaning summary"))
display(pd.DataFrame([{
    "rows_before_clean": rows_before_clean,
    "rows_after_clean": rows_after_clean,
    "rows_dropped_empty_transaction": rows_before_clean - rows_after_clean,
    "duplicate_order_id_before_merge": dup_orders_before
}]))

display(Markdown("### Sample cleaned transactions"))
display(transactions_clean_df.head())

### Cleaning summary

,rows_before_clean,rows_after_clean,rows_dropped_empty_transaction,duplicate_order_id_before_merge
0,96478,96478,0,0


### Sample cleaned transactions

,order_id,transaction_items,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,[cool_stuff],1
1,00018f77f2f0320c557190d7a144bdd3,[pet_shop],1
2,000229ec398224ef6ca0657da4fc703e,[furniture_decor],1
3,00024acbcdf0a6daa1e931b038114c75,[perfumery],1
4,00042b26cf59d7ce69dfabb4e55b4fd9,[garden_tools],1


## Cell 4: Thống kê chẩn đoán transaction dataset

**Mục tiêu của cell này**
- Chẩn đoán chất lượng transaction trước khi build basket matrix.
- Tách rõ:
  1. transaction sạch ban đầu,
  2. transaction sau khi loại `unknown`,
  3. transaction thực sự dùng cho mining (`item_count >= 2`).

**Vì sao cell này quan trọng**
- Đây là cell cho biết **coverage thật sự của market basket mining**.
- Nếu coverage quá thấp, rules vẫn có thể chạy nhưng sức mạnh thống kê/biz insight sẽ bị giới hạn.
- Cell này là bằng chứng trực tiếp để viết **phần limitation** trong báo cáo mà không suy diễn.

**Đầu ra mong đợi**
- Bảng mô tả độ dài transaction
- Số lượng transaction bị loại
- Tần suất item
- Mẫu transaction cuối cùng dùng cho mining
- Cảnh báo rõ nếu mining subset quá nhỏ


In [4]:
# =========================
# TRANSACTION DIAGNOSTICS + MINING SUBSET
# =========================
def remove_unknown_items(items):
    cleaned = []
    for item in items:
        if item is None:
            continue
        item_str = str(item).strip()
        if item_str == "" or item_str.lower() in {"nan", "none", "null", "unknown"}:
            continue
        cleaned.append(item_str)
    return sorted(set(cleaned))

transaction_length_summary = transactions_clean_df["item_count"].describe().to_frame().T
transaction_length_summary.index = ["transaction_length_raw"]

item_frequency_df_raw = (
    transactions_clean_df["transaction_items"]
    .explode()
    .value_counts()
    .reset_index()
)
item_frequency_df_raw.columns = ["item", "frequency"]

transactions_diagnostic_df = transactions_clean_df.copy()
transactions_diagnostic_df["transaction_items_no_unknown"] = (
    transactions_diagnostic_df["transaction_items"].apply(remove_unknown_items)
)
transactions_diagnostic_df["item_count_no_unknown"] = (
    transactions_diagnostic_df["transaction_items_no_unknown"].apply(len)
)

transactions_after_unknown_df = transactions_diagnostic_df[
    ["order_id", "transaction_items_no_unknown", "item_count_no_unknown"]
].rename(
    columns={
        "transaction_items_no_unknown": "transaction_items",
        "item_count_no_unknown": "item_count",
    }
).copy()

transactions_after_unknown_df = transactions_after_unknown_df[
    transactions_after_unknown_df["item_count"] > 0
].copy()

transactions_mining_df = transactions_after_unknown_df[
    transactions_after_unknown_df["item_count"] >= MIN_ITEMS_FOR_MINING
].copy()

if transactions_mining_df.empty:
    raise ValueError(
        "Sau khi loại 'unknown' và giữ basket >= 2 items, "
        "không còn transaction để mining. Cần đổi grain transaction ở notebook preprocessing."
    )

item_frequency_df = (
    transactions_mining_df["transaction_items"]
    .explode()
    .value_counts()
    .reset_index()
)
item_frequency_df.columns = ["item", "frequency"]

transaction_length_mining_summary = transactions_mining_df["item_count"].describe().to_frame().T
transaction_length_mining_summary.index = ["transaction_length_mining"]

single_item_orders_before = int((transactions_clean_df["item_count"] == 1).sum())
multi_item_orders_before = int((transactions_clean_df["item_count"] >= 2).sum())

single_item_orders_after_unknown = int((transactions_after_unknown_df["item_count"] == 1).sum())
multi_item_orders_after_unknown = int((transactions_after_unknown_df["item_count"] >= 2).sum())

mining_summary_df = pd.DataFrame([{
    "n_transactions_input": int(len(transactions_clean_df)),
    "n_transactions_after_remove_unknown": int(len(transactions_after_unknown_df)),
    "n_transactions_used_for_mining": int(len(transactions_mining_df)),
    "n_excluded_single_or_empty": int(len(transactions_after_unknown_df) - len(transactions_mining_df)),
    "avg_items_per_transaction_input": round(float(transactions_clean_df["item_count"].mean()), 4),
    "avg_items_after_remove_unknown": round(float(transactions_after_unknown_df["item_count"].mean()), 4),
    "avg_items_used_for_mining": round(float(transactions_mining_df["item_count"].mean()), 4),
    "median_items_used_for_mining": round(float(transactions_mining_df["item_count"].median()), 4),
    "max_items_used_for_mining": int(transactions_mining_df["item_count"].max()),
    "single_item_orders_before": single_item_orders_before,
    "multi_item_orders_before": multi_item_orders_before,
    "single_item_orders_after_unknown": single_item_orders_after_unknown,
    "multi_item_orders_after_unknown": multi_item_orders_after_unknown,
    "unique_items_used_for_mining": int(item_frequency_df["item"].nunique())
}])

display(Markdown("### Thống kê độ dài transaction (input)"))
display(transaction_length_summary)

display(Markdown("### Thống kê độ dài transaction (mining subset)"))
display(transaction_length_mining_summary)

display(Markdown("### Tóm tắt mining subset"))
display(mining_summary_df)

mining_coverage_pct = round(
    100.0 * len(transactions_mining_df) / max(len(transactions_after_unknown_df), 1), 4
)

mining_quality_note = (
    "Coverage mining rất thấp; rules vẫn đủ cho đồ án / UI / báo cáo nhưng không nên overclaim "
    "là đại diện mạnh cho toàn bộ hệ giao dịch."
    if mining_coverage_pct < 5.0 else
    "Coverage mining ở mức chấp nhận được cho diễn giải rules trên grain hiện tại."
)

mining_quality_df = pd.DataFrame([{
    "mining_coverage_pct_after_remove_unknown": mining_coverage_pct,
    "quality_note": mining_quality_note
}])

display(Markdown("### Cảnh báo chất lượng mining"))
display(mining_quality_df)

display(Markdown("### Top 20 items phổ biến nhất trên mining subset"))
display(item_frequency_df.head(20))

display(Markdown("### Sample transactions dùng cho mining"))
display(transactions_mining_df.head())

### Thống kê độ dài transaction (input)

,count,mean,std,min,25%,50%,75%,max
transaction_length_raw,96478.0,1.008271,0.092607,1.0,1.0,1.0,1.0,3.0


### Thống kê độ dài transaction (mining subset)

,count,mean,std,min,25%,50%,75%,max
transaction_length_mining,723.0,2.020747,0.142634,2.0,2.0,2.0,2.0,3.0


### Tóm tắt mining subset

...

,mining_coverage_pct_after_remove_unknown,quality_note
0,0.7599,Coverage mining rất thấp; rules vẫn đủ cho đồ ...


### Top 20 items phổ biến nhất trên mining subset

,item,frequency
0,furniture_decor,202
1,bed_bath_table,197
2,housewares,102
3,baby,93
4,garden_tools,71
5,health_beauty,69
6,sports_leisure,67
7,cool_stuff,66
8,toys,50
9,home_confort,50


### Sample transactions dùng cho mining

,order_id,transaction_items,item_count
72,002f98c0f7efd42638ed6100ca699b42,"[consoles_games, toys]",2
132,005d9a5423d47281ac463a968b3936fb,"[baby, toys]",2
479,014405982914c2cde2796ddcf0b8703d,"[perfumery, sports_leisure]",2
625,01b1a7fdae9ad1837d6ab861705a1fa5,"[bed_bath_table, housewares]",2
676,01cce1175ac3c4a450e3a0f856d02734,"[garden_tools, stationery]",2


## Cell 5: Xây dựng basket matrix cho FP-Growth

**Mục tiêu của cell này**
- Biến danh sách transaction thành **ma trận one-hot** để dùng cho `mlxtend.frequent_patterns.fpgrowth`.

**Logic thực hiện**
1. Lấy danh sách `transaction_items` từ mining subset.
2. Dùng `TransactionEncoder()` để biến thành basket matrix nhị phân.
3. Tính `basket_density` để biết ma trận thưa đến mức nào.
4. Sinh `support_grid_effective` dựa trên **số lần xuất hiện tuyệt đối** trên tập mining hiện tại, tránh đặt threshold thiếu cơ sở.

**Đầu ra mong đợi**
- `basket_df`
- `basket_summary_df`
- `support_grid_effective` hợp lệ để dùng cho threshold search ở cell sau


In [5]:
# =========================
# BUILD BASKET MATRIX
# =========================
transactions_list = transactions_mining_df["transaction_items"].tolist()

te = TransactionEncoder()
te_array = te.fit(transactions_list).transform(transactions_list)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

basket_density = basket_df.values.mean()

n_transactions_for_mining = len(transactions_list)
support_grid_effective = sorted({
    round(max(count / n_transactions_for_mining, 0.001), 6)
    for count in ABSOLUTE_SUPPORT_CANDIDATES
    if count <= n_transactions_for_mining
} | {0.01, 0.02, 0.03, 0.05})

support_grid_effective = [x for x in support_grid_effective if 0 < x < 1]

if not support_grid_effective:
    support_grid_effective = [round(max(1 / max(n_transactions_for_mining, 1), 0.001), 6)]

basket_summary_df = pd.DataFrame([{
    "basket_rows": basket_df.shape[0],
    "basket_cols": basket_df.shape[1],
    "basket_density": round(float(basket_density), 6),
    "support_grid_effective": ", ".join(map(lambda x: f"{x:.6f}", support_grid_effective)),
    "confidence_grid": ", ".join(map(lambda x: f"{x:.2f}", CONFIDENCE_GRID))
}])

display(Markdown("### Basket matrix summary"))
display(basket_summary_df)

display(Markdown("### Basket matrix sample"))
display(basket_df.head())

### Basket matrix summary

,basket_rows,basket_cols,basket_density,support_grid_effective,confidence_grid
0,723,61,0.033127,"0.002766, 0.004149, 0.006916, 0.010000, 0.0110...","0.05, 0.10, 0.20, 0.30, 0.40, 0.50"


### Basket matrix sample

,air_conditioning,art,arts_and_craftmanship,audio,auto,baby,bed_bath_table,books_general_interest,books_technical,christmas_supplies,...,perfumery,pet_shop,signaling_and_security,small_appliances,sports_leisure,stationery,tablets_printing_image,telephony,toys,watches_gifts
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


## Cell 6: Tìm threshold tốt nhất và sinh frequent itemsets, association rules

**Mục tiêu của cell này**
- Tìm bộ `min_support` / `min_confidence` hợp lý nhất cho grain hiện tại.
- Sinh ra:
  - `frequent_itemsets`
  - `rules_ui_df`
  - `rules_report_df`
  - `top_rules_by_lift_df`

**Chiến lược chọn threshold ở bản hoàn thiện này**
- Không chỉ ưu tiên **nhiều rule**.
- Mà ưu tiên:
  1. **rules usable cho UI**
  2. **rules đủ mạnh để đưa vào báo cáo**
  3. vẫn giữ được **Top 10 rules theo lift** cho phần tiến độ / phụ lục

**Tách rõ 2 nhóm kết quả**
- **Top 10 rules theo lift**: phù hợp cho khám phá / preview / báo cáo tiến độ, nhưng có thể có `support_count` thấp.
- **Top 5 rules report-ready**: ưu tiên cân bằng giữa `support`, `confidence`, `lift` để an toàn hơn khi diễn giải trong báo cáo cuối kỳ.

**Đầu ra mong đợi**
- threshold search table
- best params
- final itemsets/rules
- bảng rule dùng cho preview và bảng rule dùng cho report


In [6]:
# =========================
# GRID SEARCH CHO MIN_SUPPORT / MIN_CONFIDENCE
# =========================
def stringify_itemset(value):
    return ", ".join(sorted(map(str, value)))

def prepare_rules_dataframe(rules_input: pd.DataFrame, n_transactions: int) -> pd.DataFrame:
    if rules_input is None or len(rules_input) == 0:
        return pd.DataFrame()

    work = rules_input.copy()
    work["antecedent_len"] = work["antecedents"].apply(len)
    work["consequent_len"] = work["consequents"].apply(len)
    work["antecedents_str"] = work["antecedents"].apply(stringify_itemset)
    work["consequents_str"] = work["consequents"].apply(stringify_itemset)
    work["rule_str"] = work["antecedents_str"] + " → " + work["consequents_str"]
    work = work.replace([np.inf, -np.inf], np.nan)

    numeric_cols = [
        "antecedent support", "consequent support",
        "support", "confidence", "lift",
        "representativity", "leverage", "conviction",
        "zhangs_metric", "jaccard", "certainty", "kulczynski"
    ]
    existing_numeric_cols = [c for c in numeric_cols if c in work.columns]
    for col in existing_numeric_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    essential_cols = [c for c in ["support", "confidence", "lift"] if c in work.columns]
    if essential_cols:
        work = work.dropna(subset=essential_cols).copy()

    work["support_count"] = (work["support"] * n_transactions).round().astype(int)

    # Loại rules chứa unknown để tránh UI bẩn
    if "rule_str" in work.columns:
        work = work[
            ~work["rule_str"].astype(str).str.contains(r"\bunknown\b", case=False, na=False)
        ].copy()

    return work.reset_index(drop=True)

def filter_rules_for_ui(rules_input: pd.DataFrame) -> pd.DataFrame:
    if rules_input is None or len(rules_input) == 0:
        return pd.DataFrame()

    work = rules_input.copy()
    if "lift" in work.columns:
        work = work[work["lift"] > UI_MIN_LIFT].copy()
    if "confidence" in work.columns:
        work = work[work["confidence"] >= UI_MIN_CONFIDENCE].copy()
    if "support_count" in work.columns:
        work = work[work["support_count"] >= UI_MIN_SUPPORT_COUNT].copy()

    work = work.sort_values(
        ["support_count", "support", "lift", "confidence", "antecedent_len", "consequent_len"],
        ascending=[False, False, False, False, True, True]
    ).reset_index(drop=True)
    return work

def filter_rules_for_report(rules_input: pd.DataFrame) -> pd.DataFrame:
    if rules_input is None or len(rules_input) == 0:
        return pd.DataFrame()

    work = rules_input.copy()
    if "confidence" in work.columns:
        work = work[work["confidence"] >= REPORT_MIN_CONFIDENCE].copy()
    if "support_count" in work.columns:
        work = work[work["support_count"] >= REPORT_MIN_SUPPORT_COUNT].copy()

    work = work.sort_values(
        ["support_count", "support", "confidence", "lift", "antecedent_len", "consequent_len"],
        ascending=[False, False, False, False, True, True]
    ).reset_index(drop=True)
    return work

grid_rows = []

for min_support in support_grid_effective:
    frequent_itemsets_tmp = fpgrowth(
        basket_df,
        min_support=min_support,
        use_colnames=True
    )

    n_itemsets = len(frequent_itemsets_tmp)

    for min_conf in CONFIDENCE_GRID:
        if n_itemsets == 0:
            grid_rows.append({
                "min_support": min_support,
                "min_confidence": min_conf,
                "n_frequent_itemsets": 0,
                "n_rules_total": 0,
                "n_rules_ui": 0,
                "n_rules_report_ready": 0,
                "avg_lift_ui": np.nan,
                "avg_confidence_ui": np.nan,
                "avg_support_count_ui": np.nan,
                "max_lift_ui": np.nan
            })
            continue

        rules_tmp = association_rules(
            frequent_itemsets_tmp,
            metric="confidence",
            min_threshold=min_conf
        )

        rules_tmp = prepare_rules_dataframe(rules_tmp, n_transactions_for_mining)
        rules_ui_tmp = filter_rules_for_ui(rules_tmp)
        rules_report_tmp = filter_rules_for_report(rules_ui_tmp)

        grid_rows.append({
            "min_support": min_support,
            "min_confidence": min_conf,
            "n_frequent_itemsets": int(n_itemsets),
            "n_rules_total": int(len(rules_tmp)),
            "n_rules_ui": int(len(rules_ui_tmp)),
            "n_rules_report_ready": int(len(rules_report_tmp)),
            "avg_lift_ui": rules_ui_tmp["lift"].mean() if len(rules_ui_tmp) > 0 else np.nan,
            "avg_confidence_ui": rules_ui_tmp["confidence"].mean() if len(rules_ui_tmp) > 0 else np.nan,
            "avg_support_count_ui": rules_ui_tmp["support_count"].mean() if len(rules_ui_tmp) > 0 else np.nan,
            "max_lift_ui": rules_ui_tmp["lift"].max() if len(rules_ui_tmp) > 0 else np.nan
        })

support_grid_results_df = pd.DataFrame(grid_rows)

grid_display_df = support_grid_results_df.sort_values(
    [
        "n_rules_report_ready",
        "n_rules_ui",
        "avg_support_count_ui",
        "avg_lift_ui",
        "n_frequent_itemsets",
        "min_support",
        "min_confidence",
    ],
    ascending=[False, False, False, False, False, True, True]
).reset_index(drop=True)

display(Markdown("### Kết quả search threshold"))
display(grid_display_df)

valid_grid = support_grid_results_df[
    (support_grid_results_df["n_rules_ui"] > 0)
].copy()

preferred_grid = valid_grid[
    valid_grid["n_rules_report_ready"] >= PREFERRED_MIN_RULES
].copy()

if not preferred_grid.empty:
    best_row = preferred_grid.sort_values(
        [
            "n_rules_report_ready",
            "n_rules_ui",
            "avg_support_count_ui",
            "avg_lift_ui",
            "n_frequent_itemsets",
            "min_support",
            "min_confidence",
        ],
        ascending=[False, False, False, False, False, True, True]
    ).iloc[0]
    best_params = {
        "min_support": float(best_row["min_support"]),
        "min_confidence": float(best_row["min_confidence"]),
        "reason": (
            "Ưu tiên cấu hình tạo được nhiều rules report-ready "
            f"(support_count >= {REPORT_MIN_SUPPORT_COUNT}, confidence >= {REPORT_MIN_CONFIDENCE}) "
            "nhưng vẫn giữ được số rules usable cho UI."
        )
    }
elif not valid_grid.empty:
    best_row = valid_grid.sort_values(
        [
            "n_rules_ui",
            "avg_support_count_ui",
            "avg_lift_ui",
            "n_frequent_itemsets",
            "min_support",
            "min_confidence",
        ],
        ascending=[False, False, False, False, True, True]
    ).iloc[0]
    best_params = {
        "min_support": float(best_row["min_support"]),
        "min_confidence": float(best_row["min_confidence"]),
        "reason": (
            "Không đủ rules report-ready; chọn cấu hình có nhiều rules usable cho UI nhất "
            "và support_count trung bình tốt hơn."
        )
    }
else:
    best_params = {
        "min_support": float(support_grid_effective[0]),
        "min_confidence": float(CONFIDENCE_GRID[0]),
        "reason": (
            "Grid không sinh được rules usable; fallback về ngưỡng thấp nhất để export diagnostics. "
            "Khi đó cần xem lại grain transaction ở notebook preprocessing."
        )
    }

print("Best params:", best_params)

# -------------------------
# FIT FINAL
# -------------------------
frequent_itemsets = fpgrowth(
    basket_df,
    min_support=best_params["min_support"],
    use_colnames=True
)

if len(frequent_itemsets) > 0:
    rules_df_raw = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=best_params["min_confidence"]
    )
else:
    rules_df_raw = pd.DataFrame()

n_rules_raw = int(len(rules_df_raw))

# -------------------------
# CHUẨN HÓA FREQUENT ITEMSETS
# -------------------------
if not frequent_itemsets.empty:
    frequent_itemsets = frequent_itemsets.copy()
    frequent_itemsets["itemset_size"] = frequent_itemsets["itemsets"].apply(len)
    frequent_itemsets["itemsets_str"] = frequent_itemsets["itemsets"].apply(stringify_itemset)
    frequent_itemsets["support_count"] = (
        frequent_itemsets["support"] * n_transactions_for_mining
    ).round().astype(int)

    frequent_itemsets = frequent_itemsets[
        ~frequent_itemsets["itemsets_str"].astype(str).str.contains(r"\bunknown\b", case=False, na=False)
    ].copy()

    frequent_itemsets = frequent_itemsets.sort_values(
        ["support_count", "support", "itemset_size", "itemsets_str"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

# -------------------------
# CHUẨN HÓA RULES
# -------------------------
rules_df_all = prepare_rules_dataframe(rules_df_raw, n_transactions_for_mining)
rules_ui_df = filter_rules_for_ui(rules_df_all)

if rules_ui_df.empty and not rules_df_all.empty:
    # fallback mềm để không làm hỏng integration/UI
    rules_ui_df = rules_df_all.sort_values(
        ["support_count", "support", "lift", "confidence"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

rules_report_df = filter_rules_for_report(rules_ui_df)

if rules_report_df.empty and not rules_ui_df.empty:
    rules_report_df = rules_ui_df.sort_values(
        ["support_count", "support", "confidence", "lift"],
        ascending=[False, False, False, False]
    ).head(TOP_RULES_FOR_REPORT).copy()

top_rules_by_lift_df = pd.DataFrame()
if not rules_ui_df.empty:
    top_rules_by_lift_df = rules_ui_df.sort_values(
        ["lift", "confidence", "support", "support_count"],
        ascending=[False, False, False, False]
    ).head(TOP_RULES_BY_LIFT).copy()

report_rules_display = pd.DataFrame()
if not rules_report_df.empty:
    report_rules_display = rules_report_df[
        [
            "antecedents_str",
            "consequents_str",
            "support",
            "support_count",
            "confidence",
            "lift",
            "leverage",
            "conviction",
            "antecedent_len",
            "consequent_len",
            "rule_str",
        ]
    ].head(TOP_RULES_FOR_REPORT).copy()

lift_rules_display = pd.DataFrame()
if not top_rules_by_lift_df.empty:
    lift_rules_display = top_rules_by_lift_df[
        [
            "rule_str",
            "support",
            "support_count",
            "confidence",
            "lift",
            "leverage",
            "conviction",
            "antecedent_len",
            "consequent_len",
        ]
    ].copy()

display(Markdown("### Top 10 rules theo lift (phù hợp báo cáo tiến độ Tuần 2)"))
display(
    lift_rules_display
    if not lift_rules_display.empty
    else pd.DataFrame({"message": ["Không có rules usable theo threshold hiện tại."]})
)

display(Markdown("### Top 5 rules report-ready (ưu tiên support/confidence/lift cân bằng)"))
display(
    report_rules_display
    if not report_rules_display.empty
    else pd.DataFrame({"message": ["Không có rules report-ready; cần xem lại threshold hoặc grain transaction."]})
)

rule_usage_note_df = pd.DataFrame([
    {
        "nhom_rule": "Top 10 theo lift",
        "muc_dich": "Khám phá / phụ lục / báo cáo tiến độ",
        "luu_y": "Có thể có support_count thấp; không dùng toàn bộ để kết luận chính."
    },
    {
        "nhom_rule": "Top 5 report-ready",
        "muc_dich": "Bảng chính trong báo cáo cuối kỳ",
        "luu_y": "Ưu tiên support/confidence/lift cân bằng để diễn giải an toàn hơn."
    }
])

display(Markdown("### Ghi chú sử dụng rules cho preview và báo cáo"))
display(rule_usage_note_df)


### Kết quả search threshold

,min_support,min_confidence,n_frequent_itemsets,n_rules_total,n_rules_ui,n_rules_report_ready,avg_lift_ui,avg_confidence_ui,avg_support_count_ui,max_lift_ui
0,0.002766,0.05,161,146,96,39,5.045505,0.238762,7.541667,72.300000
1,0.002766,0.10,161,88,70,39,6.065322,0.299110,8.871429,72.300000
2,0.004149,0.05,102,92,62,39,3.230273,0.233464,10.580645,18.075000
3,0.004149,0.10,102,63,49,39,3.511988,0.273801,11.816327,18.075000
4,0.006916,0.05,63,51,35,31,3.202163,0.260876,15.571429,18.075000
5,0.006916,0.10,63,42,31,31,3.320093,0.284591,16.387097,18.075000
6,0.002766,0.20,161,49,43,23,7.647910,0.397878,10.906977,72.300000
7,0.004149,0.20,102,32,28,23,3.863095,0.374548,15.678571,18.075000
8,0.010000,0.05,48,36,22,20,2.218735,0.269600,21.000000,4.610435
9,0.011065,0.05,48,36,22,20,2.218735,0.269600,21.000000,4.610435


Best params: {'min_support': 0.002766, 'min_confidence': 0.05, 'reason': 'Ưu tiên cấu hình tạo được nhiều rules report-ready (support_count >= 5, confidence >= 0.1) nhưng vẫn giữ được số rules usable cho UI.'}


### Top 10 rules theo lift (phù hợp báo cáo tiến độ Tuần 2)

,rule_str,support,support_count,confidence,lift,leverage,conviction,antecedent_len,consequent_len
62,fashion_childrens_clothes → fashion_bags_acces...,0.002766,2,1.000000,72.300000,0.002728,NaN,1,1
63,fashion_bags_accessories → fashion_childrens_c...,0.002766,2,0.200000,72.300000,0.002728,1.246542,1,1
64,market_place → books_general_interest,0.002766,2,0.166667,24.100000,0.002651,1.191701,1,1
65,books_general_interest → market_place,0.002766,2,0.400000,24.100000,0.002651,1.639004,1,1
27,audio → watches_gifts,0.008299,6,1.000000,18.075000,0.007840,NaN,1,1
28,watches_gifts → audio,0.008299,6,0.150000,18.075000,0.007840,1.166707,1,1
51,office_furniture → furniture_living_room,0.004149,3,0.200000,9.640000,0.003719,1.224066,1,1
52,furniture_living_room → office_furniture,0.004149,3,0.200000,9.640000,0.003719,1.224066,1,1
66,construction_tools_safety → home_construction,0.002766,2,0.285714,7.945055,0.002418,1.349654,1,1
67,home_construction → construction_tools_safety,0.002766,2,0.076923,7.945055,0.002418,1.072845,1,1


### Top 5 rules report-ready (ưu tiên support/confidence/lift cân bằng)

,antecedents_str,consequents_str,support,support_count,confidence,lift,leverage,conviction,antecedent_len,consequent_len,rule_str
0,bed_bath_table,furniture_decor,0.096819,70,0.355330,1.271800,0.020691,1.117794,1,1,bed_bath_table → furniture_decor
1,furniture_decor,bed_bath_table,0.096819,70,0.346535,1.271800,0.020691,1.113332,1,1,furniture_decor → bed_bath_table
2,home_confort,bed_bath_table,0.059474,43,0.860000,3.156244,0.040631,5.196601,1,1,home_confort → bed_bath_table
3,bed_bath_table,home_confort,0.059474,43,0.218274,3.156244,0.040631,1.190755,1,1,bed_bath_table → home_confort
4,cool_stuff,baby,0.027663,20,0.303030,2.355816,0.015920,1.250226,1,1,cool_stuff → baby


### Ghi chú sử dụng rules cho preview và báo cáo

,nhom_rule,muc_dich,luu_y
0,Top 10 theo lift,Khám phá / phụ lục / báo cáo tiến độ,Có thể có support_count thấp; không dùng toàn ...
1,Top 5 report-ready,Bảng chính trong báo cáo cuối kỳ,Ưu tiên support/confidence/lift cân bằng để di...


## Cell 7: Lưu output, tạo summary cuối cùng và sanity check

**Mục tiêu của cell này**
- Ghi toàn bộ artifact cuối ra đúng thư mục chuẩn.
- Tạo `fpgrowth_final_summary.json` để hỗ trợ:
  - điền báo cáo,
  - debug integration,
  - kiểm tra nhanh chất lượng mining.
- Chạy `sanity check` để xác nhận file output không rỗng và contract đầu ra còn nguyên.

**Các file chính sẽ xuất ra**
- `frequent_itemsets.csv`
- `association_rules.csv` (**rules usable cho UI**)
- `top_association_rules_preview.csv` (**Top 10 theo lift**)
- `top_association_rules_report.csv` (**Top 5 report-ready**)
- `top_frequent_itemsets_preview.csv`
- `transaction_item_frequency.csv`
- `fpgrowth_threshold_search.csv`
- `fpgrowth_final_summary.json`

**Điểm cần đọc đúng khi viết báo cáo**
- Không dùng toàn bộ raw top-lift rules để kết luận business.
- Ưu tiên `top_association_rules_report.csv` cho bảng chính trong báo cáo.
- Dùng `fpgrowth_final_summary.json` để lấy số lượng itemsets/rules, coverage, threshold cuối và note limitation.


In [7]:
# =========================
# SAVE OUTPUTS + FINAL SUMMARY + SANITY CHECK
# =========================
frequent_itemsets_path = METRIC_DIR / "frequent_itemsets.csv"
association_rules_path = METRIC_DIR / "association_rules.csv"
grid_search_path = METRIC_DIR / "fpgrowth_threshold_search.csv"
top_rules_path = PRED_DIR / "top_association_rules_preview.csv"
top_rules_report_path = METRIC_DIR / "top_association_rules_report.csv"
top_itemsets_path = PRED_DIR / "top_frequent_itemsets_preview.csv"
item_frequency_path = METRIC_DIR / "transaction_item_frequency.csv"

# -------------------------
# SAVE SEARCH DIAGNOSTICS
# -------------------------
support_grid_results_to_save = support_grid_results_df.copy()
numeric_cols_grid = [
    "min_support", "min_confidence",
    "avg_lift_ui", "avg_confidence_ui",
    "avg_support_count_ui", "max_lift_ui"
]
for col in numeric_cols_grid:
    if col in support_grid_results_to_save.columns:
        support_grid_results_to_save[col] = pd.to_numeric(
            support_grid_results_to_save[col], errors="coerce"
        ).round(6)

support_grid_results_to_save.to_csv(grid_search_path, index=False)
item_frequency_df.to_csv(item_frequency_path, index=False)

# -------------------------
# SAVE FREQUENT ITEMSETS
# -------------------------
n_frequent_itemsets_saved = 0

if not frequent_itemsets.empty:
    frequent_itemsets_to_save = frequent_itemsets.copy()

    numeric_cols_itemsets = ["support", "support_count"]
    for col in numeric_cols_itemsets:
        if col in frequent_itemsets_to_save.columns:
            frequent_itemsets_to_save[col] = pd.to_numeric(
                frequent_itemsets_to_save[col], errors="coerce"
            ).round(6)

    if "itemsets" in frequent_itemsets_to_save.columns:
        frequent_itemsets_to_save = frequent_itemsets_to_save.drop(columns=["itemsets"])

    n_frequent_itemsets_saved = int(len(frequent_itemsets_to_save))
    frequent_itemsets_to_save.to_csv(frequent_itemsets_path, index=False)
    frequent_itemsets_to_save.head(20).copy().to_csv(top_itemsets_path, index=False)
else:
    pd.DataFrame(columns=["support", "support_count", "itemset_size", "itemsets_str"]).to_csv(
        frequent_itemsets_path, index=False
    )
    pd.DataFrame(columns=["support", "support_count", "itemset_size", "itemsets_str"]).to_csv(
        top_itemsets_path, index=False
    )

# -------------------------
# SAVE RULES (UI-READY FILE CHÍNH)
# -------------------------
n_rules_ui_saved = 0
n_rules_report_saved = 0

if not rules_ui_df.empty:
    rules_to_save = rules_ui_df.copy()

    numeric_cols_rules = [
        "antecedent support", "consequent support",
        "support", "support_count", "confidence", "lift",
        "representativity", "leverage", "conviction",
        "zhangs_metric", "jaccard", "certainty", "kulczynski"
    ]
    for col in numeric_cols_rules:
        if col in rules_to_save.columns:
            rules_to_save[col] = pd.to_numeric(
                rules_to_save[col], errors="coerce"
            ).round(6)

    drop_cols = [c for c in ["antecedents", "consequents"] if c in rules_to_save.columns]
    if drop_cols:
        rules_to_save = rules_to_save.drop(columns=drop_cols)

    rules_to_save = rules_to_save.head(MAX_RULES_TO_SAVE).copy()
    n_rules_ui_saved = int(len(rules_to_save))
    rules_to_save.to_csv(association_rules_path, index=False)

    top_rules_preview = top_rules_by_lift_df.copy()
    numeric_cols_preview = [
        "support", "support_count", "confidence", "lift",
        "leverage", "conviction"
    ]
    for col in numeric_cols_preview:
        if col in top_rules_preview.columns:
            top_rules_preview[col] = pd.to_numeric(
                top_rules_preview[col], errors="coerce"
            ).round(6)

    top_rules_preview.to_csv(top_rules_path, index=False)
else:
    pd.DataFrame(columns=[
        "rule_str", "support", "support_count", "confidence", "lift",
        "leverage", "conviction", "antecedent_len", "consequent_len"
    ]).to_csv(association_rules_path, index=False)

    pd.DataFrame(columns=[
        "rule_str", "support", "support_count", "confidence", "lift",
        "leverage", "conviction", "antecedent_len", "consequent_len"
    ]).to_csv(top_rules_path, index=False)

# -------------------------
# SAVE REPORT-READY TOP RULES
# -------------------------
if not report_rules_display.empty:
    report_rules_to_save = report_rules_display.copy()
    for col in ["support", "support_count", "confidence", "lift", "leverage", "conviction"]:
        if col in report_rules_to_save.columns:
            report_rules_to_save[col] = pd.to_numeric(
                report_rules_to_save[col], errors="coerce"
            ).round(6)

    n_rules_report_saved = int(len(report_rules_to_save))
    report_rules_to_save.to_csv(top_rules_report_path, index=False)
else:
    pd.DataFrame(columns=[
        "antecedents_str", "consequents_str", "support", "support_count",
        "confidence", "lift", "leverage", "conviction", "rule_str"
    ]).to_csv(top_rules_report_path, index=False)

# -------------------------
# FINAL SUMMARY
# -------------------------
mining_coverage_pct = round(
    100.0 * len(transactions_mining_df) / max(len(transactions_after_unknown_df), 1), 4
)

final_summary = {
    "transactions_source_file": str(transactions_source_path),
    "transaction_grain": "order_level_category",
    "mining_subset_rule": "remove_unknown_then_keep_item_count_gte_2",
    "min_items_for_mining": int(MIN_ITEMS_FOR_MINING),
    "n_transactions_raw": int(len(transactions_raw)),
    "n_transactions_clean": int(len(transactions_clean_df)),
    "n_transactions_after_remove_unknown": int(len(transactions_after_unknown_df)),
    "n_transactions_used_for_mining": int(len(transactions_mining_df)),
    "mining_coverage_pct_after_remove_unknown": float(mining_coverage_pct),
    "n_excluded_single_or_empty_after_remove_unknown": int(len(transactions_after_unknown_df) - len(transactions_mining_df)),
    "n_unique_items": int(item_frequency_df["item"].nunique()),
    "avg_items_per_transaction_input": float(round(transactions_clean_df["item_count"].mean(), 4)),
    "avg_items_after_remove_unknown": float(round(transactions_after_unknown_df["item_count"].mean(), 4)),
    "avg_items_used_for_mining": float(round(transactions_mining_df["item_count"].mean(), 4)),
    "median_items_used_for_mining": float(round(transactions_mining_df["item_count"].median(), 4)),
    "max_items_used_for_mining": int(transactions_mining_df["item_count"].max()),
    "basket_rows": int(basket_df.shape[0]),
    "basket_cols": int(basket_df.shape[1]),
    "basket_density": float(round(basket_df.values.mean(), 6)),
    "support_grid_effective": [float(x) for x in support_grid_effective],
    "confidence_grid": [float(x) for x in CONFIDENCE_GRID],
    "best_min_support": float(best_params["min_support"]),
    "best_min_confidence": float(best_params["min_confidence"]),
    "best_params_reason": str(best_params.get("reason", "")),
    "ui_rule_filters": {
        "min_lift": float(UI_MIN_LIFT),
        "min_confidence": float(UI_MIN_CONFIDENCE),
        "min_support_count": int(UI_MIN_SUPPORT_COUNT),
    },
    "report_rule_filters": {
        "min_confidence": float(REPORT_MIN_CONFIDENCE),
        "min_support_count": int(REPORT_MIN_SUPPORT_COUNT),
    },
    "n_rules_raw_before_cleaning": int(n_rules_raw),
    "n_rules_usable_for_ui": int(n_rules_ui_saved),
    "n_rules_report_ready": int(n_rules_report_saved),
    "n_rules_final": int(n_rules_ui_saved),
    "n_frequent_itemsets_final": int(n_frequent_itemsets_saved),
    "top_5_rules_report": report_rules_display.head(TOP_RULES_FOR_REPORT).to_dict(orient="records") if not report_rules_display.empty else [],
    "saved_files": {
        "frequent_itemsets": str(frequent_itemsets_path),
        "association_rules": str(association_rules_path),
        "threshold_search": str(grid_search_path),
        "top_rules_preview": str(top_rules_path),
        "top_rules_report": str(top_rules_report_path),
        "top_itemsets_preview": str(top_itemsets_path),
        "item_frequency": str(item_frequency_path),
    },
    "report_note": (
        "Notebook này giữ grain order-level category basket theo tài liệu hiện tại. "
        "Top 10 theo lift dùng cho tiến độ; Top 5 report-ready ưu tiên support/confidence/lift cân bằng."
    ),
    "mining_quality_warning": (
        "Coverage mining dưới 5%. Kết quả phù hợp cho đồ án / demo / UI nhưng không nên diễn giải là đại diện mạnh cho toàn bộ hệ giao dịch."
        if mining_coverage_pct < 5.0 else
        "Coverage mining ở mức chấp nhận được trên grain hiện tại."
    ),
    "rule_usage_guidance": {
        "preview_rules": "top_association_rules_preview.csv dùng cho preview / phụ lục / tiến độ; có thể chứa rule support thấp.",
        "report_rules": "top_association_rules_report.csv dùng cho bảng chính trong báo cáo cuối kỳ."
    },
    "affected_downstream_consumers": [
        "08_integration_check.ipynb (nếu đọc summary/rule files)",
        "Streamlit/UI sử dụng association_rules.csv hoặc top_association_rules_preview.csv"
    ],
    "explicitly_not_modified_by_this_notebook": [
        "01_eda_data_structure_check.ipynb",
        "02_data_preprocessing_master_dataset.ipynb",
        "03_classification_pipeline.ipynb",
        "04_regression_pipeline.ipynb",
        "05_clustering_rfm.ipynb",
        "06_surprise_recommendation (1).ipynb"
    ],
}

summary_path = METRIC_DIR / "fpgrowth_final_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

# -------------------------
# QUICK FILE CHECK
# -------------------------
produced_files = [
    frequent_itemsets_path,
    association_rules_path,
    grid_search_path,
    top_rules_path,
    top_rules_report_path,
    top_itemsets_path,
    item_frequency_path,
    summary_path,
]

file_check_df = pd.DataFrame({
    "file_path": [str(p) for p in produced_files],
    "exists": [p.exists() for p in produced_files]
})

display(Markdown("### Final summary"))
display(pd.DataFrame([final_summary]))

display(Markdown("### Produced files"))
display(file_check_df)

impact_scope_df = pd.DataFrame([
    {
        "nhom": "Files được lưu ra bởi notebook 07",
        "chi_tiet": ", ".join([p.name for p in produced_files])
    },
    {
        "nhom": "Files có thể bị ảnh hưởng logic downstream",
        "chi_tiet": "08_integration_check.ipynb / UI nếu đọc association_rules.csv, top_association_rules_preview.csv, fpgrowth_final_summary.json"
    },
    {
        "nhom": "Files không bị sửa bởi notebook 07",
        "chi_tiet": "01, 02, 03, 04, 05, 06 notebooks giữ nguyên; notebook 07 chỉ ghi artifact riêng của FP-Growth"
    }
])

display(Markdown("### Phạm vi ảnh hưởng / không ảnh hưởng"))
display(impact_scope_df)

# -------------------------
# SANITY CHECK
# -------------------------
sanity_df = pd.DataFrame({
    "check": [
        "transactions_clean_not_empty",
        "transactions_mining_not_empty",
        "basket_df_not_empty",
        "item_frequency_not_empty",
        "frequent_itemsets_file_exists",
        "association_rules_file_exists",
        "summary_json_exists",
        "rules_ui_not_empty",
        "rules_report_not_empty",
    ],
    "passed": [
        len(transactions_clean_df) > 0,
        len(transactions_mining_df) > 0,
        basket_df.shape[0] > 0 and basket_df.shape[1] > 0,
        len(item_frequency_df) > 0,
        frequent_itemsets_path.exists(),
        association_rules_path.exists(),
        summary_path.exists(),
        n_rules_ui_saved > 0,
        n_rules_report_saved > 0,
    ]
})

display(Markdown("### Sanity check"))
display(sanity_df)

# -------------------------
# FINAL PREVIEW
# -------------------------
if not frequent_itemsets.empty:
    display(Markdown("### Top 10 frequent itemsets cuối cùng"))
    display(
        frequent_itemsets[["support", "support_count", "itemset_size", "itemsets_str"]]
        .head(10)
        .copy()
    )

if not lift_rules_display.empty:
    display(Markdown("### Top 10 rules theo lift"))
    display(lift_rules_display.head(TOP_RULES_BY_LIFT))

if not report_rules_display.empty:
    display(Markdown("### Top 5 rules report-ready"))
    display(report_rules_display.head(TOP_RULES_FOR_REPORT))
else:
    display(Markdown("### Kết luận"))
    display(pd.DataFrame({
        "message": [
            "Không sinh được rules report-ready. Khi đó cần quay lại notebook preprocessing để xem lại grain transaction."
        ]
    }))

print("Đã lưu:", summary_path)

### Final summary

,transactions_source_file,transaction_grain,mining_subset_rule,min_items_for_mining,n_transactions_raw,n_transactions_clean,n_transactions_after_remove_unknown,n_transactions_used_for_mining,mining_coverage_pct_after_remove_unknown,n_excluded_single_or_empty_after_remove_unknown,...,n_rules_report_ready,n_rules_final,n_frequent_itemsets_final,top_5_rules_report,saved_files,report_note,mining_quality_warning,rule_usage_guidance,affected_downstream_consumers,explicitly_not_modified_by_this_notebook
0,..\data\processed\transactions_df.parquet,order_level_category,remove_unknown_then_keep_item_count_gte_2,2,96478,96478,95146,723,0.7599,94423,...,5,96,161,"[{'antecedents_str': 'bed_bath_table', 'conseq...",{'frequent_itemsets': '..\artifacts\metrics\fr...,Notebook này giữ grain order-level category ba...,Coverage mining dưới 5%. Kết quả phù hợp cho đ...,{'preview_rules': 'top_association_rules_previ...,[08_integration_check.ipynb (nếu đọc summary/r...,"[01_eda_data_structure_check.ipynb, 02_data_pr..."


### Produced files

,file_path,exists
0,..\artifacts\metrics\frequent_itemsets.csv,True
1,..\artifacts\metrics\association_rules.csv,True
2,..\artifacts\metrics\fpgrowth_threshold_search...,True
3,..\artifacts\predictions\top_association_rules...,True
4,..\artifacts\metrics\top_association_rules_rep...,True
5,..\artifacts\predictions\top_frequent_itemsets...,True
6,..\artifacts\metrics\transaction_item_frequenc...,True
7,..\artifacts\metrics\fpgrowth_final_summary.json,True


### Phạm vi ảnh hưởng / không ảnh hưởng

...

### Top 10 rules theo lift

,rule_str,support,support_count,confidence,lift,leverage,conviction,antecedent_len,consequent_len
62,fashion_childrens_clothes → fashion_bags_acces...,0.002766,2,1.000000,72.300000,0.002728,NaN,1,1
63,fashion_bags_accessories → fashion_childrens_c...,0.002766,2,0.200000,72.300000,0.002728,1.246542,1,1
64,market_place → books_general_interest,0.002766,2,0.166667,24.100000,0.002651,1.191701,1,1
65,books_general_interest → market_place,0.002766,2,0.400000,24.100000,0.002651,1.639004,1,1
27,audio → watches_gifts,0.008299,6,1.000000,18.075000,0.007840,NaN,1,1
28,watches_gifts → audio,0.008299,6,0.150000,18.075000,0.007840,1.166707,1,1
51,office_furniture → furniture_living_room,0.004149,3,0.200000,9.640000,0.003719,1.224066,1,1
52,furniture_living_room → office_furniture,0.004149,3,0.200000,9.640000,0.003719,1.224066,1,1
66,construction_tools_safety → home_construction,0.002766,2,0.285714,7.945055,0.002418,1.349654,1,1
67,home_construction → construction_tools_safety,0.002766,2,0.076923,7.945055,0.002418,1.072845,1,1


### Top 5 rules report-ready

,antecedents_str,consequents_str,support,support_count,confidence,lift,leverage,conviction,antecedent_len,consequent_len,rule_str
0,bed_bath_table,furniture_decor,0.096819,70,0.355330,1.271800,0.020691,1.117794,1,1,bed_bath_table → furniture_decor
1,furniture_decor,bed_bath_table,0.096819,70,0.346535,1.271800,0.020691,1.113332,1,1,furniture_decor → bed_bath_table
2,home_confort,bed_bath_table,0.059474,43,0.860000,3.156244,0.040631,5.196601,1,1,home_confort → bed_bath_table
3,bed_bath_table,home_confort,0.059474,43,0.218274,3.156244,0.040631,1.190755,1,1,bed_bath_table → home_confort
4,cool_stuff,baby,0.027663,20,0.303030,2.355816,0.015920,1.250226,1,1,cool_stuff → baby


Đã lưu: ..\artifacts\metrics\fpgrowth_final_summary.json


## CHECK-IN — FILE 07: `07_fpgrowth_rules.ipynb`

### A. THIẾT LẬP BAN ĐẦU
- [x] Hoàn thành import thư viện cho notebook FP-Growth
- [x] Hoàn thành cấu hình cảnh báo
- [x] Hoàn thành khai báo cấu hình chung cho notebook
- [x] Hoàn thành khai báo:
  - `BASE_DIR`
  - `PROCESSED_DIR`
  - `ARTIFACT_DIR`
  - `METRIC_DIR`
  - `PRED_DIR`
- [x] Hoàn thành khai báo tham số:
  - `TRANSACTIONS_SOURCE_FILE`
  - `MIN_ITEMS_FOR_MINING`
  - `ABSOLUTE_SUPPORT_CANDIDATES`
  - `CONFIDENCE_GRID`
  - `UI_MIN_LIFT`
  - `UI_MIN_CONFIDENCE`
  - `UI_MIN_SUPPORT_COUNT`

---

### B. ĐỌC DỮ LIỆU GIAO DỊCH
- [x] Hoàn thành đọc `transactions_df.parquet`
- [x] Hoàn thành hiển thị sample dữ liệu giao dịch đầu vào
- [x] Hoàn thành in `transactions_df shape`
- [x] Hoàn thành kiểm tra cột bắt buộc:
  - `order_id`
  - `transaction_items`

---

### C. LÀM SẠCH VÀ CHUẨN HÓA `transaction_items`
- [x] Hoàn thành chuẩn hóa `transaction_items` về dạng danh sách
- [x] Hoàn thành loại bỏ giá trị null / rỗng trong `transaction_items`
- [x] Hoàn thành loại bỏ phần tử trùng lặp trong từng transaction
- [x] Hoàn thành chuẩn hóa item về dạng string
- [x] Hoàn thành loại bỏ item `unknown` / item không hợp lệ
- [x] Hoàn thành tính `item_count`
- [x] Hoàn thành tạo `transactions_clean_df`
- [x] Hoàn thành tạo `transactions_mining_df`
- [x] Hoàn thành áp dụng rule giữ transaction có số item đủ điều kiện mining
- [x] Hoàn thành hiển thị sample transaction sau làm sạch

---

### D. THỐNG KÊ CHẨN ĐOÁN TRANSACTION DATASET
- [x] Hoàn thành thống kê số transaction raw
- [x] Hoàn thành thống kê số transaction sạch
- [x] Hoàn thành thống kê số transaction sau khi loại `unknown`
- [x] Hoàn thành thống kê số transaction dùng thật cho mining
- [x] Hoàn thành thống kê coverage mining
- [x] Hoàn thành thống kê số transaction bị loại vì rỗng / single-item
- [x] Hoàn thành tính phân bố `item_count`
- [x] Hoàn thành tính tần suất xuất hiện của từng item
- [x] Hoàn thành tạo `item_frequency_df`
- [x] Hoàn thành hiển thị top item frequency
- [x] Hoàn thành hiển thị sample transaction nhiều item

---

### E. XÂY DỰNG BASKET MATRIX
- [x] Hoàn thành chuyển `transactions_mining_df` thành `transactions_list`
- [x] Hoàn thành dùng `TransactionEncoder`
- [x] Hoàn thành tạo `basket_df`
- [x] Hoàn thành tính `basket_density`
- [x] Hoàn thành tính `n_transactions_for_mining`
- [x] Hoàn thành xây dựng `support_grid_effective`
- [x] Hoàn thành tạo `basket_summary_df`
- [x] Hoàn thành hiển thị basket matrix summary
- [x] Hoàn thành hiển thị sample của `basket_df`

---

### F. GRID SEARCH CHO `min_support` / `min_confidence`
- [x] Hoàn thành định nghĩa hàm `stringify_itemset()`
- [x] Hoàn thành định nghĩa hàm `prepare_rules_dataframe()`
- [x] Hoàn thành định nghĩa hàm `filter_rules_for_ui()`
- [x] Hoàn thành định nghĩa hàm `filter_rules_for_report()`
- [x] Hoàn thành khảo sát nhiều giá trị `min_support`
- [x] Hoàn thành khảo sát nhiều giá trị `min_confidence`
- [x] Hoàn thành chạy `fpgrowth()` cho từng cấu hình
- [x] Hoàn thành chạy `association_rules()` cho từng cấu hình
- [x] Hoàn thành tính cho từng cấu hình:
  - `n_frequent_itemsets`
  - `n_rules_total`
  - `n_rules_ui`
  - `n_rules_report_ready`
  - `avg_lift_ui`
  - `avg_confidence_ui`
  - `avg_support_count_ui`
  - `max_lift_ui`
- [x] Hoàn thành tạo `support_grid_results_df`
- [x] Hoàn thành hiển thị bảng grid search threshold

---

### G. CHỌN CẤU HÌNH FP-GROWTH CUỐI CÙNG
- [x] Hoàn thành chọn `best_support`
- [x] Hoàn thành chọn `best_confidence`
- [x] Hoàn thành huấn luyện FP-Growth với cấu hình cuối
- [x] Hoàn thành sinh `frequent_itemsets`
- [x] Hoàn thành sinh `association_rules`
- [x] Hoàn thành chuẩn hóa bảng rules
- [x] Hoàn thành tạo rules phục vụ UI
- [x] Hoàn thành tạo rules phục vụ report
- [x] Hoàn thành tạo preview top itemsets
- [x] Hoàn thành tạo preview top rules

---

### H. XỬ LÝ VÀ LỌC RULES
- [x] Hoàn thành thêm các cột:
  - `antecedent_len`
  - `consequent_len`
  - `antecedents_str`
  - `consequents_str`
  - `rule_str`
  - `support_count`
- [x] Hoàn thành loại rules chứa `unknown`
- [x] Hoàn thành lọc rules cho UI theo:
  - `lift`
  - `confidence`
  - `support_count`
- [x] Hoàn thành lọc rules cho report
- [x] Hoàn thành sắp xếp rules theo tiêu chí hiển thị
- [x] Hoàn thành tạo `rules_ui_ready`
- [x] Hoàn thành tạo `rules_report_ready`

---

### I. TẠO BẢNG KẾT QUẢ CUỐI CÙNG
- [x] Hoàn thành tạo bảng `frequent_itemsets`
- [x] Hoàn thành tạo bảng `association_rules`
- [x] Hoàn thành tạo top preview cho itemsets
- [x] Hoàn thành tạo top preview cho rules
- [x] Hoàn thành hiển thị preview itemsets cuối
- [x] Hoàn thành hiển thị preview rules cuối

---

### J. LƯU OUTPUT FILE
- [x] Hoàn thành lưu `frequent_itemsets.csv`
- [x] Hoàn thành lưu `association_rules.csv`
- [x] Hoàn thành lưu `fpgrowth_threshold_search.csv`
- [x] Hoàn thành lưu `top_association_rules_preview.csv`
- [x] Hoàn thành lưu `top_association_rules_report.csv`
- [x] Hoàn thành lưu `top_frequent_itemsets_preview.csv`
- [x] Hoàn thành lưu `transaction_item_frequency.csv`

---

### K. TẠO FINAL SUMMARY
- [x] Hoàn thành tạo `fpgrowth_final_summary`
- [x] Hoàn thành tổng hợp các thông tin:
  - `transactions_source_file`
  - `transaction_grain`
  - `mining_subset_rule`
  - `min_items_for_mining`
  - `n_transactions_raw`
  - `n_transactions_clean`
  - `n_transactions_after_remove_unknown`
  - `n_transactions_used_for_mining`
  - `mining_coverage_pct_after_remove_unknown`
  - `n_excluded_single_or_empty_after_remove_unknown`
  - `best_min_support`
  - `best_min_confidence`
  - `n_rules_report_ready`
  - `n_rules_final`
  - `n_frequent_itemsets_final`
  - `top_5_rules_report`
  - `saved_files`
  - `report_note`
  - `mining_quality_warning`
  - `rule_usage_guidance`
  - `affected_downstream_consumers`
  - `explicitly_not_modified_by_this_notebook`
- [x] Hoàn thành hiển thị final summary
- [x] Hoàn thành lưu `fpgrowth_final_summary.json`

---

### L. KIỂM TRA FILE OUTPUT
- [x] Hoàn thành kiểm tra sự tồn tại của các file output chính
- [x] Hoàn thành tạo bảng `file_path / exists`
- [x] Hoàn thành hiển thị danh sách file đã lưu

---

### M. GHI CHÚ ẢNH HƯỞNG DOWNSTREAM
- [x] Hoàn thành tạo bảng mô tả:
  - files được lưu ra bởi notebook 07
  - files có thể bị ảnh hưởng logic downstream
  - files không bị sửa bởi notebook 07

---

### N. SANITY CHECK CUỐI
- [x] Hoàn thành kiểm tra:
  - `transactions_clean_not_empty`
  - `transactions_mining_not_empty`
  - `basket_df_not_empty`
  - `item_frequency_not_empty`
  - `frequent_itemsets_file_exists`
  - `association_rules_file_exists`
  - `summary_json_exists`
  - `rules_ui_not_empty`
  - `rules_report_not_empty`
- [x] Hoàn thành hiển thị bảng sanity check cuối

---

### O. KẾT QUẢ ĐẦU RA CHÍNH CỦA NOTEBOOK
- [x] Hoàn thành phần **đọc dữ liệu giao dịch**
- [x] Hoàn thành phần **làm sạch và chuẩn hóa `transaction_items`**
- [x] Hoàn thành phần **thống kê chẩn đoán transaction dataset**
- [x] Hoàn thành phần **xây dựng basket matrix**
- [x] Hoàn thành phần **grid search cho `min_support` / `min_confidence`**
- [x] Hoàn thành phần **chọn cấu hình FP-Growth cuối cùng**
- [x] Hoàn thành phần **sinh frequent itemsets**
- [x] Hoàn thành phần **sinh association rules**
- [x] Hoàn thành phần **lọc rules cho UI**
- [x] Hoàn thành phần **lọc rules cho report**
- [x] Hoàn thành phần **lưu output files**
- [x] Hoàn thành phần **tạo final summary JSON**
- [x] Hoàn thành phần **kiểm tra file output**
- [x] Hoàn thành phần **sanity check cuối**

### Kết luận cuối:
Notebook `07_fpgrowth_rules.ipynb` đã **hoàn thành** toàn bộ các hạng mục chính thuộc phạm vi **FP-Growth Pipeline, Transaction Cleaning, Basket Matrix Construction, Threshold Grid Search, Frequent Itemsets, Association Rules, Rule Filtering for UI/Report, Output Saving, Final Summary, và Sanity Check** của riêng file này.